In [ ]:
# Instala a biblioteca especializada em extração de PDFs
!pip install pdfplumber -q
print("✅ pdfplumber instalado com sucesso!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 58.3 MB/s eta 0:00:00
✅ pdfplumber instalado com sucesso!


In [ ]:
import requests
import pandas as pd
from google.colab import userdata

# ─────────────────────────────────────────────
# 1. AUTENTICAÇÃO SEGURA (COLAB SECRETS)
# ─────────────────────────────────────────────
try:
    # O script busca a chave no cofre do Colab, cumprindo o requisito de segurança
    API_TOKEN = userdata.get("PORTAL_TRANSPARENCIA_TOKEN")
    print("🔑 Token carregado com sucesso do ambiente seguro do Colab!")

    # Prepara o cabeçalho seguro para quando for fazer a requisição na API
    HEADERS = {
        "chave-api-dados": API_TOKEN,
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    }
except userdata.SecretNotFoundError:
    print("❌ Erro: O segredo 'PORTAL_TRANSPARENCIA_TOKEN' não foi encontrado.")
    print("Verifique a aba lateral 'Secrets' (ícone de chave).")
    API_TOKEN = None

🔑 Token carregado com sucesso do ambiente seguro do Colab!


In [ ]:
import pdfplumber
import pandas as pd

# ─────────────────────────────────────────────
# 2. LEITURA E EXTRAÇÃO "BLINDADA" DA REMUME
# ─────────────────────────────────────────────
NOME_ARQUIVO_PDF = "remume_versao_2024.pdf"

def extrair_todas_tabelas_blindado(caminho_arquivo):
    todas_as_linhas = []
    nomes_das_colunas = []

    try:
        print(f"📄 Abrindo o documento: {caminho_arquivo}...")

        with pdfplumber.open(caminho_arquivo) as pdf:
            total_paginas = len(pdf.pages)
            print(f"✅ Documento lido! Extraindo dados (Páginas 6 a {total_paginas})...\n")

            for numero_pagina in range(5, total_paginas):
                pagina_atual = pdf.pages[numero_pagina]
                tabelas = pagina_atual.extract_tables()

                if not tabelas:
                    continue

                tabela_principal = tabelas[0]

                # Na primeira página de tabela (índice 5), definimos o "Padrão Ouro" do cabeçalho
                if numero_pagina == 5:
                    nomes_das_colunas = tabela_principal[0]
                    # Limpa as quebras de linha dos nomes das colunas
                    nomes_das_colunas = [str(c).replace('\n', ' ') if c else f"Coluna_{i}" for i, c in enumerate(nomes_das_colunas)]
                    linhas_dados = tabela_principal[1:]
                else:
                    linhas_dados = tabela_principal

                tamanho_esperado = len(nomes_das_colunas)

                for linha in linhas_dados:
                    # Verifica se é uma linha válida (não totalmente nula)
                    if not linha or all(celula is None or str(celula).strip() == "" for celula in linha):
                        continue

                    # Se a linha for mais curta, adicionamos espaços em branco
                    if len(linha) < tamanho_esperado:
                        linha_ajustada = linha + [""] * (tamanho_esperado - len(linha))
                    # Se a linha for mais longa (a coluna fantasma), cortamos o excesso
                    else:
                        linha_ajustada = linha[:tamanho_esperado]

                    todas_as_linhas.append(linha_ajustada)

            df_completo = pd.DataFrame(todas_as_linhas, columns=nomes_das_colunas)

            # Limpa quebras de linha dentro dos textos dos medicamentos para ficar bonito
            df_completo = df_completo.replace('\n', ' ', regex=True)
            df_completo = df_completo.fillna("")
            print("📊 Extração Concluída com Sucesso! Visão geral:")
            print(f"Total de itens na lista municipal: {len(df_completo)}")
            display(df_completo.head(10))

    except Exception as e:
        print(f"❌ Ocorreu um erro: {e}")

# Executa o script
extrair_todas_tabelas_blindado(NOME_ARQUIVO_PDF)

📄 Abrindo o documento: remume_versao_2024.pdf...
✅ Documento lido! Extraindo dados (Páginas 6 a 28)...

📊 Extração Concluída com Sucesso! Visão geral:
Total de itens na lista municipal: 264


,Item,Medicamento,Apresentação,Local de acesso,Coluna_4,Classificação e tipo de receita
0,01,Acetilcisteína,600 mg granulado envelopes com 5 g,- Farmácias Distritais,,- Expectorante - Receituário simples
1,02,Aciclovir,200 mg Comprimido,- Farmácias Distritais - Farmácia Estratégica,,- Antiviral - Receituário simples
2,03,Aciclovir,50 mg/g Creme bisnaga,,- Farmácias Distritais,- Antiviral - Receituário simples
3,,,,,- Farmácia Estratégica,
4,,,,,- Dispensários nas UBS,
5,04,Ácido acetilsalicílico,100 mg Comprimido,- Farmácias Distritais - Dispensários nas UBS,,- Antitrombótico - Receituário simples
6,05,Ácido fólico,5 mg Comprimido,- Farmácias Distritais - Farmácia Estratégica ...,,- Vitamina - Receituário simples
7,06,Ácido fólico,"0,2 mg/ml Solução oral Frasco",- Farmácias Distritais - Dispensários nas UBS,,- Vitamina - Receituário simples
8,07,Ácido valpróico (valproato de sódio) (Controla...,"50 mg/ml (equivalente a 57,624mg ácido valprói...",- Farmácias Distritais,,- Anticonvulsivante - Receituário de Controle ...
9,08,Ácido valpróico (valproato de sódio) (Controla...,500 mg (equivalente a 576 mg ácido valpróico) ...,- Farmácias Distritais,,- Anticonvulsivante - Receituário de Controle ...
